# Sampling module [AFTER PYLINT ANALYSIS]

In [1]:
def default_params(): 
    return {
        'current_model': 'M1',
        'gpu': False,
        'quantization': 'none', #['none',"int4", "int8", "float32", "float16"]
        'dataset': {
            'path': '/workspaces/CodeSmells/datax/code_smells/generation/dataset',
            #['greedy_search', 'beam_search', 'sampling', 'contrastive_search', 'top_k_sampling', 'top_p_sampling']
            'decoding_strategy': 'top_p_sampling',
            'content_column': 'code',
            'sampling_size': 500,
        },
        'logging_path': '/workspaces/CodeSmells/datax/code_smells/logs/generation', 
        'callbacks_dir' : '/workspaces/CodeSmells/datax/code_smells/callbacks/generation',
        'cache_dir': '/workspaces/CodeSmells/datax/hugging_face_cache',
        'causal_models': {
            'M1' : 'codellama/CodeLlama-7b-hf', #https://huggingface.co/codellama/CodeLlama-7b-hf, 
            'M2' : 'mistralai/Mistral-7B-v0.3', #https://huggingface.co/mistralai/Mistral-7B-v0.3,
            'M3' : 'microsoft/Phi-3.5-mini-instruct', #https://huggingface.co/microsoft/Phi-3.5-mini-instruct 
            'M4' : 'Qwen/Qwen2.5-Coder-7B', #https://huggingface.co/Qwen/Qwen2.5-Coder-7B
            'M5' : 'facebook/incoder-6B', #https://huggingface.co/facebook/incoder-6B
            'M6' : 'bigcode/starcoder2-7b', #https://huggingface.co/bigcode/starcoder2-7b 
            'M7' : 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B', #https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Llama-8B
            'M8' : 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', #https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
        },
    }
params = default_params()


#### Imports

In [2]:
import pandas as pd
import os
import time
import numpy as np
import torch
import gc
import seaborn as sns
from scipy import stats
from statistics import NormalDist
import matplotlib.pyplot as plt

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

In [4]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

#### Dataset

In [5]:
df_dataset = pd.read_json(f"{params['dataset']['path']}/{params['current_model']}_q_{params['quantization']}/{params['dataset']['decoding_strategy']}_{params['dataset']['sampling_size']}.json")

In [6]:
sampled_df = df_dataset.groupby('s_msg_id', group_keys=False).apply(lambda x: x.sample(n=min(len(x), params['dataset']['sampling_size']), random_state=42))
sampled_df = sampled_df.reset_index(drop=True)


/tmp/ipykernel_1007708/3748123824.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_df = df_dataset.groupby('s_msg_id', group_keys=False).apply(lambda x: x.sample(n=min(len(x), params['dataset']['sampling_size']), random_state=42))


In [7]:
sampled_df.to_json(f"{params['dataset']['path']}/{params['current_model']}_q_{params['quantization']}/{params['dataset']['decoding_strategy']}_resampled_{params['dataset']['sampling_size']}.json", index=False)